## tl;dr
현재 데이터 검증과 성과 인증은 다릅니다. 읽기 전용 감사파일·회귀검사 결과를 재현합니다. 원천 수집이나 배포는 실행하지 않습니다.


## Context & Methods
프로젝트 Python 의존성이 필요합니다. Jupyter 패키지는 설치되어 있지 않아 아래 코드셀을 Python으로 순서대로 실행해 확인했습니다.
### Key Assumptions
기준일은 감사파일에 기록된 2026-09-07입니다. 실시간 값이 아닌 저장된 감사 스냅샷이며 API 공급자별 모집단이 다릅니다. 가격 이력 길이가 당시 선정 입력의 완비성을 뜻하지 않습니다.


In [ ]:
from pathlib import Path
import json, sqlite3, xml.etree.ElementTree as ET
from kr_quant.research.selection_ledger import read_verified
ROOT = Path('C:/Users/a4jud/kr_quant_research')
pit = read_verified(ROOT/'output/audit/pit-final-20260908.json')['payload']
edge = read_verified(ROOT/'output/audit/flow-edge-cases-fixed-20260908.json')['payload']
menu = json.loads((ROOT/'output/item2-integration-audit.json').read_text(encoding='utf-8'))
assert all(menu['checks'].values())
assert edge['probes']['official_missing_numeric_rows_accepted'] == 0
assert not edge['probes']['toss_missing_foreign_candidate_accepted']
assert edge['probes']['mixed_unit_candidates'] == 0
print('menu_checks', len(menu['checks']), 'verified_performance', pit['verified'])


## Data
공급자별 저장 종목과 현재 후보 수를 감사 JSON에서 읽습니다. 아래 SQLite 테이블은 이 감사의 소규모 메모리 투영이며 영구 DB를 수정하지 않습니다.


In [ ]:
db = sqlite3.connect(':memory:')
db.execute('CREATE TABLE stored_flow_coverage(provider TEXT, current INTEGER, stored INTEGER, expected_date TEXT)')
f = menu['flow']
d = menu['freshness']['expected_price_date']
db.executemany('INSERT INTO stored_flow_coverage VALUES (?,?,?,?)', [('토스', f['toss']['current_rows'], f['toss']['source_rows'], d), ('공식 KIS', f['official']['current_tickers'], f['official']['stored_tickers'], d)])
SQL = "SELECT provider, 100.0 * current / stored AS valid_percent, current, stored, stored-current AS excluded, expected_date FROM stored_flow_coverage ORDER BY current ASC"
rows = db.execute(SQL).fetchall()
print(rows)
db.close()


## Results
테스트 파일 간 중복을 제거해 검사 수를 계산합니다. 전체 저장소 검사라고 해석하지 않습니다.


In [ ]:
cases = {}
for name in ('reliability-combined-tests-20260908.xml', 'reliability-final-delta-tests-20260908.xml', 'reliability-web-smoke-20260908.xml'):
    for case in ET.parse(ROOT/'output/audit'/name).iter('testcase'):
        cases[(case.get('classname'), case.get('name'))] = case
assert all(c.find('failure') is None and c.find('error') is None and c.find('skipped') is None for c in cases.values())
print('unique_selected_passed', len(cases))
print({name: (r.get('rows'), r.get('duplicate_key_rows'), r.get('status')) for name,r in pit['sources'].items()})


## Takeaways
다음 단계는 최신 수급 재수집, 공식 기업행위·상폐 정산 및 과거 원본 확보, 고정 운영 선정기의 미사용 기간 검증입니다. 원천이 없는 구간을 과거 날짜로 복제하지 않습니다. 전체 테스트·브라우저 성능·공개 배포는 이번 범위의 완료 항목이 아닙니다.
